<a href="https://colab.research.google.com/github/apurvapm/Bird-Image-Recognizer-CUB/blob/main/CUB_EF~1_IPY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune EfficientNet on CUB-200-2011

Trains EfficientNet-B0 (ImageNet-pretrained) on the 200 bird species in CUB-200-2011, end to end on a Colab GPU.

**Before running:** Runtime → Change runtime type → T4 GPU.

Run cells top to bottom. Expect ~5–10 minutes for the full training run on a T4.


In [ ]:
import torch, torchvision
print("torch       :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("CUDA        :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## 1. Download CUB-200-2011 (~1.1 GB, one-time)

In [ ]:
!mkdir -p /content/data
!wget -q -c -O /content/data/CUB_200_2011.tgz https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz
!ls -lh /content/data/CUB_200_2011.tgz


In [ ]:
import tarfile, os
from pathlib import Path
CUB_ROOT = Path("/content/data/CUB_200_2011")
if not (CUB_ROOT / "images.txt").exists():
    print("Extracting...")
    with tarfile.open("/content/data/CUB_200_2011.tgz", "r:gz") as tf:
        tf.extractall("/content/data")
print("CUB files:", sorted(os.listdir(CUB_ROOT))[:8])


## 2. Dataset class + transforms

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

def _read_id_map(path, cast=str):
    out = {}
    for line in open(path):
        a, _, b = line.strip().partition(" ")
        if a:
            out[int(a)] = cast(b)
    return out

def load_class_names(root=CUB_ROOT):
    raw = _read_id_map(root / "classes.txt")
    return [raw[i + 1].split(".", 1)[1].replace("_", " ") for i in range(200)]

class CUBDataset(Dataset):
    def __init__(self, root, train=True, transform=None):
        self.root, self.transform = Path(root), transform
        paths = _read_id_map(self.root / "images.txt")
        labels = _read_id_map(self.root / "image_class_labels.txt", int)
        split = _read_id_map(self.root / "train_test_split.txt", int)
        want = 1 if train else 0
        self.samples = [
            (self.root / "images" / paths[i], labels[i] - 1)
            for i in paths if split[i] == want
        ]
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        p, y = self.samples[i]
        img = Image.open(p).convert("RGB")
        return self.transform(img) if self.transform else img, y

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])


In [ ]:
train_ds = CUBDataset(CUB_ROOT, train=True,  transform=train_tf)
val_ds   = CUBDataset(CUB_ROOT, train=False, transform=eval_tf)
class_names = load_class_names()
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Classes: {len(class_names)}")

BATCH = 32
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)


## 3. Peek at a batch

In [ ]:
import matplotlib.pyplot as plt

def unnormalize(t):
    t = t.clone()
    for c, (m, s) in enumerate(zip(MEAN, STD)):
        t[c] = t[c] * s + m
    return t.clamp(0, 1).permute(1, 2, 0).numpy()

xs, ys = next(iter(DataLoader(train_ds, batch_size=8, shuffle=True)))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, img, label in zip(axes.flat, xs, ys):
    ax.imshow(unnormalize(img))
    ax.set_title(class_names[label.item()], fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()


## 4. Build EfficientNet-B0 with a fresh 200-class head

In [ ]:
import torch.nn as nn
from torchvision import models

def build_model(num_classes=200, freeze=False):
    m = models.efficientnet_b0(weights="DEFAULT")
    in_f = m.classifier[-1].in_features
    m.classifier[-1] = nn.Linear(in_f, num_classes)
    if freeze:
        for p in m.parameters():
            p.requires_grad = False
        for p in m.classifier[-1].parameters():
            p.requires_grad = True
    return m

# With a GPU we go full fine-tune (set freeze=True for CPU / quick demo).
model = build_model(num_classes=200, freeze=False).to(device)
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {n_train:,} / {n_total:,} ({100*n_train/n_total:.1f}%)")


## 5. Train

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm

EPOCHS = 10
LR     = 3e-4
WD     = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = AdamW([p for p in model.parameters() if p.requires_grad],
                  lr=LR, weight_decay=WD)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    # ---- train ----
    model.train()
    tl = tc = tn = 0
    pbar = tqdm(train_loader, desc=f"epoch {epoch}/{EPOCHS}")
    for x, y in pbar:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        tl += loss.item() * x.size(0)
        tc += (out.argmax(1) == y).sum().item()
        tn += x.size(0)
        pbar.set_postfix(loss=f"{tl/tn:.3f}", acc=f"{tc/tn:.3f}")
    train_loss, train_acc = tl / tn, tc / tn

    # ---- validate ----
    model.eval()
    vl = vc = vn = 0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device); y = y.to(device)
            out = model(x); loss = criterion(out, y)
            vl += loss.item() * x.size(0)
            vc += (out.argmax(1) == y).sum().item()
            vn += x.size(0)
    val_loss, val_acc = vl / vn, vc / vn
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    print(f"[{epoch:02d}]  train_loss={train_loss:.3f}  train_acc={train_acc:.3f}  "
          f"val_loss={val_loss:.3f}  val_acc={val_acc:.3f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save({
            "state_dict":  model.state_dict(),
            "class_names": class_names,
            "model_name":  "efficientnet_b0",
            "val_acc":     val_acc,
            "epoch":       epoch,
        }, "/content/best.pt")
        print(f"  -> new best val_acc {val_acc:.4f}, saved /content/best.pt")

print(f"\nDone. Best val_acc = {best_acc:.4f}")


## 6. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"],   label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"],   label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); plt.show()


## 7. Top-1 / top-5 evaluation on the test set

In [ ]:
model.eval()
top1 = top5 = total = 0
with torch.no_grad():
    for x, y in tqdm(val_loader, desc="evaluating"):
        x = x.to(device); y = y.to(device)
        logits = model(x)
        _, pred5 = logits.topk(5, dim=1)
        top1 += (pred5[:, 0] == y).sum().item()
        top5 += (pred5 == y.unsqueeze(1)).any(1).sum().item()
        total += y.size(0)
print(f"Top-1 acc: {top1/total:.4f}")
print(f"Top-5 acc: {top5/total:.4f}")


## 8. Predict on an uploaded photo

In [ ]:
from google.colab import files
import io

uploaded = files.upload()  # opens a file picker
for fname, data in uploaded.items():
    img = Image.open(io.BytesIO(data)).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(x), dim=1)[0]
    top_p, top_i = probs.topk(5)

    plt.figure(figsize=(5, 5))
    plt.imshow(img); plt.axis("off"); plt.title(fname); plt.show()
    print(f"\nTop 5 predictions for {fname}:")
    for p, i in zip(top_p, top_i):
        bar = "#" * int(round(p.item() * 30))
        print(f"  {class_names[i]:<35s} {p.item():6.2%}  {bar}")


In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/cub_efficientnet"
os.makedirs(DRIVE_DIR, exist_ok=True)
shutil.copy("/content/best.pt", f"{DRIVE_DIR}/best.pt")
print(f"Saved to {DRIVE_DIR}/best.pt")


## Next steps

- Try `models.efficientnet_b2` or `b3` for higher accuracy (slower, more VRAM).
- Edit `train_tf` to add `transforms.RandAugment()` or `AutoAugment(transforms.AutoAugmentPolicy.IMAGENET)`.
- Compare `freeze=True` vs `freeze=False` accuracy and training time — great visualization of why transfer learning matters.
- Plug the saved `best.pt` into the sister `bird_id` Gradio app for a polished demo.
